In [0]:
CREATE OR REPLACE TEMP VIEW exh_rfm_scored AS
WITH params AS (
  SELECT DATE'2026-08-01' AS as_of_date
),

theatres AS (
  SELECT DISTINCT LocationID
  FROM cpx_dataanalytics_gold.customer360.cpx_d_location
  WHERE LC_Location_Type_Description = 'Cineplex Theatre'
),

visits AS (
  SELECT
    t.CDE_ID, t.VisitDateId, t.LocationId,
    p.as_of_date,
    max(to_date(d.datefull))                                  AS visit_date,
    max(CASE WHEN t.TicketFLAG = 'Yes' THEN 1 ELSE 0 END)     AS has_ticket,
    max(CASE WHEN t.ConcessionFLAG = 'Yes' THEN 1 ELSE 0 END) AS has_concession,
    sum(t.Net_Revenue)                                        AS visit_revenue
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
  JOIN theatres th ON t.LocationId = th.LocationID
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON t.VisitDateId = d.dateid
  CROSS JOIN params p
  WHERE t.CDE_ID IS NOT NULL
    AND to_date(d.datefull) >= add_months(p.as_of_date, -12)
    AND to_date(d.datefull) <  p.as_of_date
  GROUP BY t.CDE_ID, t.VisitDateId, t.LocationId, p.as_of_date
),

raw AS (
  SELECT
    CDE_ID,
    as_of_date,
    datediff(as_of_date, max(CASE WHEN has_ticket = 1 AND visit_revenue > 0 THEN visit_date END))  AS exh_recency_days,
    count_if(has_ticket = 1 AND visit_revenue > 0)                                                  AS exh_frequency,
    count_if(has_ticket = 0 AND has_concession = 1 AND visit_revenue > 0)                           AS exh_concession_only_visits,
    sum(visit_revenue)                                                                              AS exh_net_revenue,
    sum(visit_revenue) / nullif(count_if(has_ticket = 1 AND visit_revenue > 0), 0)                  AS exh_monetary
  FROM visits
  GROUP BY CDE_ID, as_of_date
  HAVING exh_frequency > 0 OR exh_concession_only_visits > 0
),

cutoffs AS (
  SELECT
    percentile_approx(exh_recency_days, array(0.25, 0.5, 0.75))                                   AS r_cut,
    percentile_approx(CASE WHEN exh_frequency >= 2 THEN exh_frequency END, array(0.3333, 0.6667)) AS f_cut,
    percentile_approx(exh_monetary, array(0.25, 0.5, 0.75))                                       AS m_cut
  FROM raw
  WHERE exh_frequency > 0
),

scored AS (
  SELECT
    r.*,
    c.r_cut, c.f_cut, c.m_cut,
    -- new: percentile cut-offs (F1 = one-timers, F2–F4 = tertiles of repeaters)
    CASE WHEN exh_frequency = 0 THEN NULL
         WHEN exh_recency_days <= c.r_cut[0] THEN 4
         WHEN exh_recency_days <= c.r_cut[1] THEN 3
         WHEN exh_recency_days <= c.r_cut[2] THEN 2 ELSE 1 END AS r_new,
    CASE WHEN exh_frequency = 0 THEN NULL
         WHEN exh_frequency = 1 THEN 1
         WHEN exh_frequency <= c.f_cut[0] THEN 2
         WHEN exh_frequency <= greatest(c.f_cut[1], c.f_cut[0] + 1) THEN 3 ELSE 4 END AS f_new,
    CASE WHEN exh_frequency = 0 THEN NULL
         WHEN exh_monetary <= c.m_cut[0] THEN 1
         WHEN exh_monetary <= c.m_cut[1] THEN 2
         WHEN exh_monetary <= c.m_cut[2] THEN 3 ELSE 4 END AS m_new,
    -- old: fixed cut-offs
    CASE WHEN exh_frequency = 0 THEN NULL
         WHEN exh_recency_days >= 180 THEN 1 WHEN exh_recency_days >= 90 THEN 2
         WHEN exh_recency_days >= 45 THEN 3 ELSE 4 END AS r_old,
    CASE WHEN exh_frequency = 0 THEN NULL
         WHEN exh_frequency > 6 THEN 4 WHEN exh_frequency > 3 THEN 3
         WHEN exh_frequency > 1 THEN 2 ELSE 1 END AS f_old,
    CASE WHEN exh_frequency = 0 THEN NULL
         WHEN exh_monetary >= 40 THEN 4 WHEN exh_monetary >= 30 THEN 3
         WHEN exh_monetary >= 20 THEN 2 ELSE 1 END AS m_old
  FROM raw r
  CROSS JOIN cutoffs c
),

seg_map AS (
  SELECT segment, explode(codes) AS code
  FROM VALUES
    ('Champions',              array('443','444')),
    ('Potential Loyalist',     array('232','233','234','242','243','244','332','333','334','342','343','344','432','433','434','442')),
    ('Value Conscious Loyals', array('231','241','331','341','431','441')),
    ('Promising',              array('221','222','223','224','321','322','323','324','421','422','423','424')),
    ('New',                    array('311','312','313','314','411','412','413','414')),
    ('At Risk',                array('121','122','123','124','131','132','133','134','141','142','143','144')),
    ('Fleeting',               array('111','112','211','212','213','214')),
    ('High Value Lapsed',      array('113','114'))
  AS v(segment, codes)
)

SELECT
  s.*,
  coalesce(o.segment, 'NA') AS old_segment,
  coalesce(n.segment, 'NA') AS new_segment
FROM scored s
LEFT JOIN seg_map o
  ON concat(CAST(s.r_old AS STRING), CAST(s.f_old AS STRING), CAST(s.m_old AS STRING)) = o.code
LEFT JOIN seg_map n
  ON concat(CAST(s.r_new AS STRING), CAST(s.f_new AS STRING), CAST(s.m_new AS STRING)) = n.code;


WITH existing AS (
  SELECT m.CDE_ID, m.Recency_Days, m.No_of_Visits, m.Net_Revenue_per_Visit, m.Net_Revenue,
         CAST(m.R_Score AS INT) AS r_ex, CAST(m.F_Score AS INT) AS f_ex, CAST(m.M_Score AS INT) AS m_ex,
         m.Segment AS seg_ex
  FROM cpx_dataanalytics_gold.customer360.c360_f_rfm_model m
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON m.SnapshotDateId = d.dateid
  WHERE to_date(d.datefull) = DATE'2026-08-01'
),
j AS (
  SELECT e.*, s.CDE_ID AS s_id, s.exh_recency_days, s.exh_frequency, s.exh_monetary, s.exh_net_revenue,
         s.r_old, s.f_old, s.m_old, s.old_segment,
         (e.CDE_ID IS NOT NULL AND s.CDE_ID IS NOT NULL) AS in_both
  FROM existing e
  FULL OUTER JOIN exh_rfm_scored s ON e.CDE_ID = s.CDE_ID
)
SELECT
  count(CDE_ID)                                                    AS n_existing,
  count(s_id)                                                      AS n_ours,
  count_if(in_both)                                                AS n_both,
  count_if(s_id IS NULL AND No_of_Visits > 0)                      AS only_existing_active,
  count_if(s_id IS NULL AND CDE_ID IS NOT NULL AND coalesce(No_of_Visits, 0) = 0) AS only_existing_inactive,
  count_if(CDE_ID IS NULL)                                         AS only_ours,
  count_if(CDE_ID IS NULL AND exh_frequency > 0)                   AS only_ours_movie,
  round(100 * avg(CASE WHEN in_both THEN CAST(No_of_Visits = exh_frequency AS INT) END), 2)             AS pct_visits_match,
  round(100 * avg(CASE WHEN in_both THEN CAST(Recency_Days = exh_recency_days AS INT) END), 2)          AS pct_recency_exact,
  round(100 * avg(CASE WHEN in_both THEN CAST(abs(Recency_Days - exh_recency_days) <= 1 AS INT) END), 2) AS pct_recency_within_1d,
  round(100 * avg(CASE WHEN in_both THEN CAST(abs(Net_Revenue_per_Visit - exh_monetary) < 0.01 AS INT) END), 2) AS pct_rpv_match,
  round(100 * avg(CASE WHEN in_both THEN CAST(abs(Net_Revenue - exh_net_revenue) < 0.01 AS INT) END), 2) AS pct_rev_match,
  round(100 * avg(CASE WHEN in_both THEN CAST(r_ex = r_old AS INT) END), 2)      AS pct_r_match,
  round(100 * avg(CASE WHEN in_both THEN CAST(f_ex = f_old AS INT) END), 2)      AS pct_f_match,
  round(100 * avg(CASE WHEN in_both THEN CAST(m_ex = m_old AS INT) END), 2)      AS pct_m_match,
  round(100 * avg(CASE WHEN in_both THEN CAST(seg_ex = old_segment AS INT) END), 2) AS pct_segment_match
FROM j;

WITH existing AS (
  SELECT m.CDE_ID, m.Net_Revenue
  FROM cpx_dataanalytics_gold.customer360.c360_f_rfm_model m
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON m.SnapshotDateId = d.dateid
  WHERE to_date(d.datefull) = DATE'2026-08-01'
),
theatres AS (
  SELECT DISTINCT LocationID
  FROM cpx_dataanalytics_gold.customer360.cpx_d_location
  WHERE LC_Location_Type_Description = 'Cineplex Theatre'
),
visits AS (
  SELECT
    t.CDE_ID, t.VisitDateId, t.LocationId,
    max(CASE WHEN t.TicketFLAG = 'Yes' THEN 1 ELSE 0 END)                        AS has_ticket,
    max(CASE WHEN t.TicketFLAG = 'Yes' AND t.Net_Revenue > 0 THEN 1 ELSE 0 END)  AS paid_ticket,
    sum(t.Net_Revenue)                                                           AS visit_revenue
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
  JOIN theatres th ON t.LocationId = th.LocationID
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON t.VisitDateId = d.dateid
  WHERE t.CDE_ID IN (SELECT CDE_ID FROM existing)
    AND to_date(d.datefull) >= DATE'2025-08-01'
    AND to_date(d.datefull) <  DATE'2026-08-01'
  GROUP BY t.CDE_ID, t.VisitDateId, t.LocationId
),
cust AS (
  SELECT
    CDE_ID,
    sum(visit_revenue)                                                               AS rev_all,
    sum(CASE WHEN visit_revenue > 0 THEN visit_revenue ELSE 0 END)                   AS rev_positive_visits,
    sum(CASE WHEN has_ticket = 1 THEN visit_revenue ELSE 0 END)                      AS rev_ticket_visits,
    sum(CASE WHEN has_ticket = 1 AND visit_revenue > 0 THEN visit_revenue ELSE 0 END) AS rev_movie_visits,
    sum(CASE WHEN paid_ticket = 1 THEN visit_revenue ELSE 0 END)                     AS rev_paid_ticket_visits
  FROM visits
  GROUP BY CDE_ID
)
SELECT
  count(*) AS n_both,
  round(100 * avg(CAST(abs(e.Net_Revenue - c.rev_all)                < 0.01 AS INT)), 2) AS pct_match_all_revenue,
  round(100 * avg(CAST(abs(e.Net_Revenue - c.rev_positive_visits)    < 0.01 AS INT)), 2) AS pct_match_positive_visits,
  round(100 * avg(CAST(abs(e.Net_Revenue - c.rev_ticket_visits)      < 0.01 AS INT)), 2) AS pct_match_ticket_visits,
  round(100 * avg(CAST(abs(e.Net_Revenue - c.rev_movie_visits)       < 0.01 AS INT)), 2) AS pct_match_movie_visits,
  round(100 * avg(CAST(abs(e.Net_Revenue - c.rev_paid_ticket_visits) < 0.01 AS INT)), 2) AS pct_match_paid_ticket_visits
FROM existing e
JOIN cust c ON e.CDE_ID = c.CDE_ID;

WITH existing AS (
  SELECT m.CDE_ID, m.Net_Revenue
  FROM cpx_dataanalytics_gold.customer360.c360_f_rfm_model m
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON m.SnapshotDateId = d.dateid
  WHERE to_date(d.datefull) = DATE'2026-08-01'
),
theatres AS (
  SELECT DISTINCT LocationID
  FROM cpx_dataanalytics_gold.customer360.cpx_d_location
  WHERE LC_Location_Type_Description = 'Cineplex Theatre'
),
lines AS (
  SELECT
    t.CDE_ID, t.Net_Revenue, t.Net_Spend, t.TicketFLAG, t.ConcessionFLAG, t.Box_Office_SundryFLAG, t.STOREFLAG,
    (t.VisitDateId       >= 20250801 AND t.VisitDateId       < 20260801) AS in_visit_window,
    (t.TransactionDateid >= 20250801 AND t.TransactionDateid < 20260801) AS in_txn_window
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
  JOIN theatres th ON t.LocationId = th.LocationID
  WHERE t.CDE_ID IN (SELECT CDE_ID FROM existing)
    AND ((t.VisitDateId       >= 20250801 AND t.VisitDateId       < 20260801)
      OR (t.TransactionDateid >= 20250801 AND t.TransactionDateid < 20260801))
),
cust AS (
  SELECT
    CDE_ID,
    coalesce(sum(CASE WHEN in_visit_window THEN Net_Revenue END), 0)                                                     AS rev_all,
    coalesce(sum(CASE WHEN in_visit_window AND (TicketFLAG = 'Yes' OR ConcessionFLAG = 'Yes') THEN Net_Revenue END), 0)  AS rev_ticket_conc_lines,
    coalesce(sum(CASE WHEN in_visit_window AND coalesce(Box_Office_SundryFLAG, 'No') <> 'Yes' THEN Net_Revenue END), 0)  AS rev_no_sundry,
    coalesce(sum(CASE WHEN in_visit_window AND coalesce(STOREFLAG, 'No') <> 'Yes' THEN Net_Revenue END), 0)              AS rev_no_store,
    coalesce(sum(CASE WHEN in_visit_window THEN Net_Spend END), 0)                                                       AS spend_all,
    coalesce(sum(CASE WHEN in_txn_window THEN Net_Revenue END), 0)                                                       AS rev_by_purchase_date
  FROM lines
  GROUP BY CDE_ID
)
SELECT
  count(*) AS n_both,
  round(100 * avg(CAST(abs(e.Net_Revenue - c.rev_all)               < 0.01 AS INT)), 2) AS pct_all_revenue,
  round(100 * avg(CAST(abs(e.Net_Revenue - c.rev_ticket_conc_lines) < 0.01 AS INT)), 2) AS pct_ticket_conc_lines,
  round(100 * avg(CAST(abs(e.Net_Revenue - c.rev_no_sundry)         < 0.01 AS INT)), 2) AS pct_no_sundry,
  round(100 * avg(CAST(abs(e.Net_Revenue - c.rev_no_store)          < 0.01 AS INT)), 2) AS pct_no_store,
  round(100 * avg(CAST(abs(e.Net_Revenue - c.spend_all)             < 0.01 AS INT)), 2) AS pct_net_spend,
  round(100 * avg(CAST(abs(e.Net_Revenue - c.rev_by_purchase_date)  < 0.01 AS INT)), 2) AS pct_purchase_date,
  -- size of the gap where all-revenue doesn't match
  percentile_approx(CASE WHEN abs(e.Net_Revenue - c.rev_all) >= 0.01 THEN c.rev_all - e.Net_Revenue END, 0.5)                                  AS median_gap_dollars,
  percentile_approx(CASE WHEN abs(e.Net_Revenue - c.rev_all) >= 0.01 THEN (c.rev_all - e.Net_Revenue) / nullif(e.Net_Revenue, 0) END, 0.5)    AS median_gap_pct
FROM existing e
JOIN cust c ON e.CDE_ID = c.CDE_ID;

WITH existing AS (
  SELECT m.CDE_ID, m.Net_Revenue
  FROM cpx_dataanalytics_gold.customer360.c360_f_rfm_model m
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON m.SnapshotDateId = d.dateid
  WHERE to_date(d.datefull) = DATE'2026-08-01'
),
theatres AS (
  SELECT DISTINCT LocationID
  FROM cpx_dataanalytics_gold.customer360.cpx_d_location
  WHERE LC_Location_Type_Description = 'Cineplex Theatre'
),
cust AS (
  SELECT
    t.CDE_ID,
    sum(t.Net_Revenue)                                                                         AS rev_all,
    sum(CASE WHEN t.Box_Office_SundryFLAG = 'Yes' THEN t.Net_Revenue ELSE 0 END)               AS rev_sundry,
    sum(CASE WHEN t.ConcessionFLAG = 'Yes' THEN t.Net_Revenue ELSE 0 END)                      AS rev_concession,
    sum(CASE WHEN t.CineClubFLAG = 'Yes' AND coalesce(t.TicketFLAG, 'No') <> 'Yes' THEN t.Net_Revenue ELSE 0 END) AS rev_cineclub_non_ticket,
    sum(CASE WHEN t.COTFLAG = 'Yes' AND coalesce(t.TicketFLAG, 'No') <> 'Yes' THEN t.Net_Revenue ELSE 0 END)      AS rev_online_non_ticket,
    sum(CASE WHEN t.Box_Office_SundryFLAG = 'Yes' AND t.COTFLAG = 'No' THEN t.Net_Revenue ELSE 0 END)             AS rev_sundry_in_person
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
  JOIN theatres th ON t.LocationId = th.LocationID
  WHERE t.CDE_ID IN (SELECT CDE_ID FROM existing)
    AND t.VisitDateId >= 20250801 AND t.VisitDateId < 20260801
  GROUP BY t.CDE_ID
),
gap AS (
  SELECT c.*, c.rev_all - e.Net_Revenue AS gap
  FROM existing e
  JOIN cust c ON e.CDE_ID = c.CDE_ID
  WHERE abs(c.rev_all - e.Net_Revenue) >= 0.01
)
SELECT
  count(*) AS n_mismatched,
  round(100 * avg(CAST(abs(gap - rev_sundry)              < 0.01 AS INT)), 2) AS pct_gap_is_sundry,
  round(100 * avg(CAST(abs(gap - rev_sundry_in_person)    < 0.01 AS INT)), 2) AS pct_gap_is_sundry_in_person,
  round(100 * avg(CAST(abs(gap - rev_concession)          < 0.01 AS INT)), 2) AS pct_gap_is_concession,
  round(100 * avg(CAST(abs(gap - rev_cineclub_non_ticket) < 0.01 AS INT)), 2) AS pct_gap_is_cineclub_non_ticket,
  round(100 * avg(CAST(abs(gap - rev_online_non_ticket)   < 0.01 AS INT)), 2) AS pct_gap_is_online_non_ticket
FROM gap;

WITH existing AS (
  SELECT m.CDE_ID, m.Net_Revenue
  FROM cpx_dataanalytics_gold.customer360.c360_f_rfm_model m
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON m.SnapshotDateId = d.dateid
  WHERE to_date(d.datefull) = DATE'2026-08-01'
),
theatres AS (
  SELECT DISTINCT LocationID
  FROM cpx_dataanalytics_gold.customer360.cpx_d_location
  WHERE LC_Location_Type_Description = 'Cineplex Theatre'
),
lbe AS (
  SELECT DISTINCT LocationID
  FROM cpx_dataanalytics_gold.customer360.cpx_d_location
  WHERE LC_Location_Type_Description = 'Restaurant/Store/Games'
),
cust AS (
  SELECT
    t.CDE_ID,
    sum(CASE WHEN th.LocationID IS NOT NULL THEN t.Net_Revenue ELSE 0 END)                               AS rev_theatre,
    sum(CASE WHEN th.LocationID IS NOT NULL AND t.ConcessionFLAG = 'Yes' THEN t.Net_Revenue ELSE 0 END)  AS rev_theatre_conc,
    sum(CASE WHEN lb.LocationID IS NOT NULL THEN t.Net_Revenue ELSE 0 END)                               AS rev_lbe
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
  LEFT JOIN theatres th ON t.LocationId = th.LocationID
  LEFT JOIN lbe lb      ON t.LocationId = lb.LocationID
  WHERE t.CDE_ID IN (SELECT CDE_ID FROM existing)
    AND t.VisitDateId >= 20250801 AND t.VisitDateId < 20260801
    AND (th.LocationID IS NOT NULL OR lb.LocationID IS NOT NULL)
  GROUP BY t.CDE_ID
)
SELECT
  count(*) AS n_both,
  round(100 * avg(CAST(abs(e.Net_Revenue - rev_theatre)                               < 0.01 AS INT)), 2) AS pct_theatre_all,
  round(100 * avg(CAST(abs(e.Net_Revenue - (rev_theatre - rev_theatre_conc))          < 0.01 AS INT)), 2) AS pct_theatre_no_conc,
  round(100 * avg(CAST(abs(e.Net_Revenue - (rev_theatre - rev_theatre_conc + rev_lbe)) < 0.01 AS INT)), 2) AS pct_theatre_no_conc_plus_lbe,
  round(100 * avg(CAST(abs(e.Net_Revenue - (rev_theatre + rev_lbe))                   < 0.01 AS INT)), 2) AS pct_theatre_all_plus_lbe
FROM existing e
JOIN cust c ON e.CDE_ID = c.CDE_ID;